In [17]:
"""
Shuffle Files — PySpark 4.0
===========================
Every wide transformation (groupBy, join, repartition, distinct…) produces
a shuffle: map tasks write partitioned files to local disk; reduce tasks
fetch the relevant partitions over the network (or locally in single-node
mode) and process them.

Cost model
  shuffle write  = serialize + sort + write to local disk (1 data file + 1 index file per upstream task)
  shuffle read   = fetch + deserialize (1 partition per downstream task)

This notebook surfaces five failure modes and their fixes:
  1. spark.sql.shuffle.partitions default (200) overhead on small data
  2. AQE — adaptive partition coalescing
  3. Shuffle stage compounding (wide op chaining)
  4. Skewed shuffle — one reducer drowns
  5. Broadcast join — eliminating the shuffle entirely
"""

import glob
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [18]:
def section(title: str) -> None:
    bar = "=" * 70
    print(f"\n{bar}\n  {title}\n{bar}\n")


def partition_sizes(df) -> list[int]:
    return df.rdd.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()


def shuffle_file_count(spark) -> int:
    """Count shuffle data files Spark has written to its local temp dir."""
    local_dir = spark.conf.get("spark.local.dir", "/tmp")
    return len(glob.glob(f"{local_dir}/**/shuffle_*.data", recursive=True))


def timed(label: str, action):
    t0 = time.perf_counter()
    result = action()
    elapsed = time.perf_counter() - t0
    print(f"{label}: {elapsed:.2f}s")
    return result


spark = (
    SparkSession.builder
    .appName("ShuffleFiles")
    .master("local[8]")
    # Start with AQE off so we see the raw shuffle behaviour;
    # we enable it explicitly in section 2.
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")  # force shuffles for joins
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("shuffle.partitions default:", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.sparkContext.defaultParallelism: ", spark.sparkContext.defaultParallelism)

shuffle.partitions default: 10
spark.sparkContext.defaultParallelism:  8


In [24]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 1. spark.sql.shuffle.partitions — THE DEFAULT 200 PROBLEM
#
#  Every shuffle stage produces spark.sql.shuffle.partitions reduce tasks.
#  The default is 200.  On small data this means 200 nearly-empty tasks,
#  200 output files, and significant scheduling overhead.
#
#  On large data 200 is often too few — partitions become huge and spill.
# ═══════════════════════════════════════════════════════════════════════════ #
section("1. spark.sql.shuffle.partitions — the default 200 problem")
print("""
  Group 50 000 rows by a key with 10 distinct values.
  Compare 200 shuffle partitions (default) vs 10 (right-sized).

  Look for:  output partition count, timing, and shuffle file count.
             190 of the 200 output partitions will be empty.
""")

df = (
    spark.range(50_000)
    .withColumn("key", F.col("id") % 10)
    .withColumn("value", F.col("id") * 2)
)

spark.conf.set("spark.sql.shuffle.partitions", "200")
before = shuffle_file_count(spark)
result_200 = df.groupBy("key").agg(F.sum("value").alias("total"))
print("200 Shuffle Partitions:")
result_200.explain("formatted")
timed("groupBy, 200 shuffle partitions", lambda: result_200.count())
sizes_200 = partition_sizes(result_200)
print(f"  output partitions : {len(sizes_200)}")
print(f"  non-empty         : {sum(1 for s in sizes_200 if s > 0)}")
print(f"  shuffle files on disk: {shuffle_file_count(spark) - before}")

spark.conf.set("spark.sql.shuffle.partitions", "10")
before = shuffle_file_count(spark)
print("\n\n10 Shuffle Partitions:")
result_10 = df.groupBy("key").agg(F.sum("value").alias("total"))
result_10.explain("formatted")
timed("groupBy, 10 shuffle partitions", lambda: result_10.count())
sizes_10 = partition_sizes(result_10)
print(f"  output partitions : {len(sizes_10)}")
print(f"  non-empty         : {sum(1 for s in sizes_10 if s > 0)}")
print(f"  shuffle files on disk: {shuffle_file_count(spark) - before}")

result_200.unpersist()
result_10.unpersist()


  1. spark.sql.shuffle.partitions — the default 200 problem


  Group 50 000 rows by a key with 10 distinct values.
  Compare 200 shuffle partitions (default) vs 10 (right-sized).

  Look for:  output partition count, timing, and shuffle file count.
             190 of the 200 output partitions will be empty.

200 Shuffle Partitions:
== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * Project (2)
         +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#2153L]
Arguments: Range (0, 50000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [2]: [(id#2153L % 10) AS key#2154L, (id#2153L * 2) AS value#2155L]
Input [1]: [id#2153L]

(3) HashAggregate [codegen id : 1]
Input [2]: [key#2154L, value#2155L]
Keys [1]: [key#2154L]
Functions [1]: [partial_sum(value#2155L)]
Aggregate Attributes [1]: [sum#2161L]
Results [2]: [key#2154L, sum#2162L]

(4) Exchange
Input [2]: [key#2154L, sum#2162L]
Arguments: hashpartitioning(key#2154L, 200), E

  output partitions : 200
  non-empty         : 10
  shuffle files on disk: 0


10 Shuffle Partitions:
== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * Project (2)
         +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#2153L]
Arguments: Range (0, 50000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [2]: [(id#2153L % 10) AS key#2154L, (id#2153L * 2) AS value#2155L]
Input [1]: [id#2153L]

(3) HashAggregate [codegen id : 1]
Input [2]: [key#2154L, value#2155L]
Keys [1]: [key#2154L]
Functions [1]: [partial_sum(value#2155L)]
Aggregate Attributes [1]: [sum#2175L]
Results [2]: [key#2154L, sum#2176L]

(4) Exchange
Input [2]: [key#2154L, sum#2176L]
Arguments: hashpartitioning(key#2154L, 10), ENSURE_REQUIREMENTS, [plan_id=2700]

(5) HashAggregate [codegen id : 2]
Input [2]: [key#2154L, sum#2176L]
Keys [1]: [key#2154L]
Functions [1]: [sum(value#2155L)]
Aggregate Attributes [1]: [sum(value#2155L)#2174L]
Results [2]: [key#2154L

DataFrame[key: bigint, total: bigint]

In [25]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 2. AQE — ADAPTIVE SHUFFLE PARTITION COALESCING
#
#  Adaptive Query Execution (spark.sql.adaptive.enabled) measures actual
#  shuffle map output sizes and coalesces small adjacent partitions before
#  the reduce stage starts.  This makes shuffle.partitions safe to set high
#  without paying the 200-empty-task penalty on small data.
#
#  Key config:
#    spark.sql.adaptive.coalescePartitions.enabled   (default true with AQE)
#    spark.sql.adaptive.advisoryPartitionSizeInBytes (default 64 MB)
# ═══════════════════════════════════════════════════════════════════════════ #
section("2. AQE — adaptive shuffle partition coalescing")
print("""
  Same query, same 200 shuffle.partitions, but AQE enabled.
  AQE inspects map output sizes and merges partitions below the advisory
  threshold before scheduling reducers.

  Look for:  AQEShuffleRead wrapping the Exchange in the plan (isFinalPlan=true).
             actual output partition count much lower than 200.
""")

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
# Lower threshold so coalescing is visible on our small dataset
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "1mb")

result_aqe = df.groupBy("key").agg(F.sum("value").alias("total"))

count = result_aqe.count()  # AQE adapts the plan during execution

# explain() after execution shows isFinalPlan=true with AQEShuffleRead
result_aqe.explain("formatted")
print(f"rows returned          : {count}")
print(f"output partitions (AQE): {result_aqe.rdd.getNumPartitions()}")

spark.conf.set("spark.sql.adaptive.enabled", "false")


  2. AQE — adaptive shuffle partition coalescing


  Same query, same 200 shuffle.partitions, but AQE enabled.
  AQE inspects map output sizes and merges partitions below the advisory
  threshold before scheduling reducers.

  Look for:  AQEShuffleRead wrapping the Exchange in the plan (isFinalPlan=true).
             actual output partition count much lower than 200.

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Range (1)


(1) Range
Output [1]: [id#2153L]
Arguments: Range (0, 50000, step=1, splits=Some(8))

(2) Project
Output [2]: [(id#2153L % 10) AS key#2154L, (id#2153L * 2) AS value#2155L]
Input [1]: [id#2153L]

(3) HashAggregate
Input [2]: [key#2154L, value#2155L]
Keys [1]: [key#2154L]
Functions [1]: [partial_sum(value#2155L)]
Aggregate Attributes [1]: [sum#2195L]
Results [2]: [key#2154L, sum#2196L]

(4) Exchange
Input [2]: [key#2154L, sum#2196L]
Arguments: hashpartitioning(key#

In [26]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 3. SHUFFLE STAGE COMPOUNDING
#
#  Each wide transformation adds a shuffle stage — an Exchange node in the
#  plan that writes files and waits for all upstream tasks to complete before
#  downstream tasks can start.  Chaining wide ops multiplies this cost.
#
#  A DAG with N wide ops has N stage boundaries and N round-trips to disk.
# ═══════════════════════════════════════════════════════════════════════════ #
section("3. Shuffle stage compounding — each wide op adds a stage")
print("""
  Compare plans for 1, 2, and 3 chained groupBys.
  Each groupBy introduces one Exchange (shuffle write) and one barrier.

  Look for:  Exchange count in the physical plan growing with each chain.
             Stage count in Spark UI grows correspondingly.
""")

spark.conf.set("spark.sql.shuffle.partitions", "8")

base = (
    spark.range(100_000)
    .withColumn("dept",   F.col("id") % 5)
    .withColumn("region", F.col("id") % 3)
    .withColumn("value",  F.col("id").cast("double"))
)

one_shuffle = base.groupBy("dept").agg(F.sum("value").alias("dept_total"))

two_shuffles = (
    base
    .groupBy("dept").agg(F.sum("value").alias("dept_total"))
    .groupBy("dept_total").agg(F.count("*").alias("freq"))
)

three_shuffles = (
    base
    .groupBy("dept", "region").agg(F.sum("value").alias("subtotal"))
    .groupBy("dept").agg(F.sum("subtotal").alias("dept_total"))
    .groupBy("dept_total").agg(F.count("*").alias("freq"))
)

print("--- 1 groupBy (1 shuffle stage) ---")
one_shuffle.explain("formatted")

print("--- 2 chained groupBys (2 shuffle stages) ---")
two_shuffles.explain("formatted")

print("--- 3 chained groupBys (3 shuffle stages) ---")
three_shuffles.explain("formatted")

t1 = timed("1 shuffle",  lambda: one_shuffle.count())
t2 = timed("2 shuffles", lambda: two_shuffles.count())
t3 = timed("3 shuffles", lambda: three_shuffles.count())


  3. Shuffle stage compounding — each wide op adds a stage


  Compare plans for 1, 2, and 3 chained groupBys.
  Each groupBy introduces one Exchange (shuffle write) and one barrier.

  Look for:  Exchange count in the physical plan growing with each chain.
             Stage count in Spark UI grows correspondingly.

--- 1 groupBy (1 shuffle stage) ---
== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * Project (2)
         +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#2198L]
Arguments: Range (0, 100000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [2]: [(id#2198L % 5) AS dept#2199L, cast(id#2198L as double) AS value#2201]
Input [1]: [id#2198L]

(3) HashAggregate [codegen id : 1]
Input [2]: [dept#2199L, value#2201]
Keys [1]: [dept#2199L]
Functions [1]: [partial_sum(value#2201)]
Aggregate Attributes [1]: [sum#2233]
Results [2]: [dept#2199L, sum#2234]

(4) Exchange
Input [2]: [dept#2199L, sum#2234]
Arguments: hashpar

In [29]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 4. SKEWED SHUFFLE — ONE REDUCER DROWNS
#
#  Hash partitioning sends all rows with the same key to the same reducer.
#  If one key dominates the dataset, one task handles the bulk of the data
#  while all others finish quickly — the job stalls waiting for the straggler.
#
#  AQE skew join optimisation detects this at runtime and splits the large
#  partition into sub-tasks, then duplicates the matching build-side data.
# ═══════════════════════════════════════════════════════════════════════════ #
section("4. Skewed shuffle — one reducer drowns")
print("""
  Build a heavily skewed dataset: key=0 gets 90% of rows.
  Join against a small lookup to preserve all rows and make the skew visible.

  Look for:  one enormous partition vs many tiny ones without AQE.
             With AQE skewJoin enabled, the hot partition is split into
             sub-tasks and partition sizes become more even.
""")

spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force shuffle join

hot   = spark.range(90_000).withColumn("key", F.lit(0))
other = spark.range(10_000).withColumn("key", (F.col("id") % 9 + 1).cast("long"))
skewed = hot.union(other).withColumn("value", F.col("key") * 2)

lookup = spark.range(10).withColumnRenamed("id", "key").withColumn("label", F.col("key").cast("string"))

join_no_aqe = skewed.join(lookup, "key")
join_no_aqe.count()
print("partition sizes without AQE (join, all rows preserved):")
print(partition_sizes(join_no_aqe))

# Enable AQE skew join
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1kb")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "1kb")

join_aqe = skewed.join(lookup, "key")
join_aqe.count()
print("\npartition sizes with AQE skew join:")
print(partition_sizes(join_aqe))

spark.conf.set("spark.sql.adaptive.enabled", "false")


  4. Skewed shuffle — one reducer drowns


  Build a heavily skewed dataset: key=0 gets 90% of rows.
  Join against a small lookup to preserve all rows and make the skew visible.

  Look for:  one enormous partition vs many tiny ones without AQE.
             With AQE skewJoin enabled, the hot partition is split into
             sub-tasks and partition sizes become more even.

partition sizes without AQE (join, all rows preserved):
[1111, 1111, 0, 2222, 3333, 92223, 0, 0]

partition sizes with AQE skew join:
[2222, 278, 278, 277, 278, 278, 278, 277, 278, 417, 416, 417, 417, 416, 416, 417, 417, 11250, 11250, 11250, 11250, 11250, 11250, 11250, 11250, 278, 278, 278, 277, 278, 278, 278, 278]


In [23]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 5. BROADCAST JOIN — ELIMINATING THE SHUFFLE ENTIRELY
#
#  When one join side is small enough to fit in executor memory, Spark can
#  broadcast it to every executor.  The large side is never shuffled —
#  each task reads its local partition and probes the in-memory hash table.
#
#  Result: zero shuffle write, zero shuffle read, no stage boundary.
# ═══════════════════════════════════════════════════════════════════════════ #
section("5. Broadcast join — eliminating the shuffle entirely")
print("""
  Join a large table (500 000 rows) against a small lookup (10 rows).
  Compare shuffle join (broadcast disabled) vs broadcast join.

  Look for:  Exchange nodes disappear from the large side in the broadcast plan.
             BroadcastExchange replaces SortMergeJoin.
             Timing difference despite single-node mode (no real network cost).
""")

spark.conf.set("spark.sql.shuffle.partitions", "8")

large = (
    spark.range(500_000)
    .withColumn("dept_id", (F.col("id") % 10).cast("int"))
    .withColumn("value",   F.col("id").cast("double"))
)
small = spark.createDataFrame(
    [(i, f"dept_{i}") for i in range(10)],
    ["dept_id", "dept_name"],
)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
shuffle_join = large.join(small, "dept_id")
print("--- shuffle join (broadcast disabled) ---")
shuffle_join.explain("formatted")
t_shuffle = timed("shuffle join", lambda: shuffle_join.count())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10 MB
broadcast_join = large.join(small, "dept_id")
print("--- broadcast join ---")
broadcast_join.explain("formatted")
t_broadcast = timed("broadcast join", lambda: broadcast_join.count())


  5. Broadcast join — eliminating the shuffle entirely


  Join a large table (500 000 rows) against a small lookup (10 rows).
  Compare shuffle join (broadcast disabled) vs broadcast join.

  Look for:  Exchange nodes disappear from the large side in the broadcast plan.
             BroadcastExchange replaces SortMergeJoin.
             Timing difference despite single-node mode (no real network cost).

--- shuffle join (broadcast disabled) ---
== Physical Plan ==
* Project (11)
+- * SortMergeJoin Inner (10)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Project (3)
   :        +- * Filter (2)
   :           +- * Range (1)
   +- * Sort (9)
      +- Exchange (8)
         +- * Filter (7)
            +- * Scan ExistingRDD (6)


(1) Range [codegen id : 1]
Output [1]: [id#2132L]
Arguments: Range (0, 500000, step=1, splits=Some(8))

(2) Filter [codegen id : 1]
Input [1]: [id#2132L]
Condition : isnotnull(cast((id#2132L % 10) as int))

(3) Project [codegen id : 1]
Output [3]: [id#2132L

In [ ]:
spark.stop()